In [1]:
import pickle
import os
import pandas as pd
import pickle
import numpy as np
import ast

In [19]:
BIOS_LABELS = "data/BIOS_LABELS_v2.csv" # this contains all of our changes without the sensitive data, but ids matching the BIOS dataset
BIOS_RAW = "../../data/biosbias2023/BIOS.pkl" # path to the downloaded BIOS.pkl

reviewed_classes = ['architect', 'surgeon', 'dentist', 'teacher', 'psychologist', 'nurse', 'photographer', 'physician', 'attorney', 'journalist', 'software_enginner', 'model']

In [ ]:
# get raw data from crawler
with (open(BIOS_RAW, "rb")) as openfile:
    raw_data = pickle.load(openfile)
df_raw = pd.DataFrame(raw_data)
df_raw['index'] = df_raw.index
df_raw = df_raw.rename(columns={'raw_title': 'auto_raw_title', 'title': 'auto_title'})
df_raw

In [ ]:
# get labels
df_label = pd.read_csv(BIOS_LABELS, sep='\t')
df_label

### Merges texts and names from crawled dataset into the label patch

In [12]:
COLUMNS_TO_MERGE = ['index', 'review', 'raw_titles', 'titles', 'valid']

merged_df = pd.merge(
    df_raw,
    df_label[COLUMNS_TO_MERGE],
    on='index',
    how='left'
)

In [13]:
def parse_titles(value):
    """Convert a cell of the 'titles' column to a list of strings."""
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        value = value.strip()
        if not value:                      # empty string -> no labels
            return []
        try:
            parsed = ast.literal_eval(value)   # handles "['a', 'b']"
            if isinstance(parsed, list):
                return [str(x) for x in parsed]
        except (ValueError, SyntaxError):
            pass
        # fallback: plain string with a delimiter, e.g. "spam, ham"
        return [part.strip() for part in value.split(',') if part.strip()]
    return []  

In [ ]:
merged_df['raw_titles'] = merged_df['raw_titles'].apply(parse_titles)
merged_df['titles'] = merged_df['titles'].apply(parse_titles)
merged_df

In [15]:
def filter_dataset(df: pd.DataFrame, classes: list, single_label=True, review_only=True, valid_only=True, remove_other_classes=True):
    if review_only:
        df = df[df['review'] == 1]
    if valid_only:
        df = df[df['valid'] == 1]

    if single_label:
        # Keep only rows whose 'titles' list has exactly one element
        df = df[df['titles'].apply(lambda t: isinstance(t, list) and len(t) == 1)]

        # Extract the single label as an int index into `classes`;
        # -1 means "class not in `classes`"
        def to_single_label(titles):
            title = titles[0]
            return classes.index(title) if title in classes else -1

        df = df.copy()
        df['label'] = df['titles'].apply(to_single_label)

        if remove_other_classes:
            df = df[df['label'] != -1]
    else:
        # One binary column per class (multi-label), 1 if the title is present
        df = df.copy()
        for cls in classes:
            df[f'label_{cls}'] = df['titles'].apply(
                lambda t: int(cls in t) if isinstance(t, list) else 0
            )

        if remove_other_classes:
            # Drop rows containing any title that is not in `classes`
            classes_set = set(classes)
            df = df[df['titles'].apply(
                lambda t: isinstance(t, list) and all(x in classes_set for x in t)
            )]

    return df

In [ ]:
# only reviewed samples, as multi-label
df = filter_dataset(merged_df, classes=reviewed_classes, single_label=False, review_only=True, valid_only=False, remove_other_classes=False)
df

In [ ]:
# only reviewed and valid samples as multi-label
df = filter_dataset(merged_df, classes=reviewed_classes, single_label=False, review_only=True, valid_only=True, remove_other_classes=False)
df

In [ ]:
# only reviewed and valid samples as single-label, remove other classes
df = filter_dataset(merged_df, classes=reviewed_classes, single_label=True, review_only=True, valid_only=True, remove_other_classes=False)
df